<a href="https://colab.research.google.com/github/t09Simi/Gen_AI_Concepts/blob/main/RAG_Lab_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Building a RAG Application — Step by Step

**For IBM "Build RAG Applications with LangChain" cert prep — Colab edition**

This notebook walks through Retrieval-Augmented Generation (RAG) from scratch. Each section explains *what* is happening and *why* — so you understand the concepts, not just the code.

## What is RAG?

LLMs are smart but limited:
- They don't know about your private documents
- Their knowledge has a cutoff date
- They hallucinate facts when they don't know something

**RAG fixes this** by retrieving relevant chunks from your documents *before* the LLM answers, so the LLM has the right context to work with.

**The pipeline:**
```
PDF → Split into chunks → Convert to embeddings → Store in vector DB
                                                          ↓
User Question → Embed question → Find similar chunks → Pass to LLM → Answer
```


---
## Step 0: Setup — Install Dependencies

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`. The embedding model runs much faster on GPU.

In [ ]:
!pip install -q --upgrade \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-google-genai \
    langchain-chroma \
    chromadb==0.5.23 \
    pypdf==5.2.0 \
    sentence-transformers==3.2.1 \
    tiktoken==0.8.0

---
## Step 1: Set Up Your API Key (Gemini)

Why Gemini? Google's free tier is generous (15 requests/min, 1M tokens/day), no credit card needed, and `gemini-1.5-flash` is fast and capable enough for RAG.

**Get your key:**
1. Go to https://aistudio.google.com/apikey
2. Click "Create API Key"
3. Copy it

**Add it to Colab securely:**
1. Click the 🔑 (key) icon in the left sidebar
2. Click "Add new secret"
3. Name: `GOOGLE_API_KEY`, Value: paste your key
4. Toggle "Notebook access" ON

**Why use Secrets instead of pasting the key in a cell?** If you share or publish the notebook, your key won't leak.

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("✅ API key loaded")

---
## Step 2: Get a Sample PDF

We'll use the **"Attention Is All You Need"** paper — the transformer paper that started modern LLMs. It's a good RAG test document because it's technical, has structure, and most LLMs know *about* it but not the exact contents.

You can replace this URL with any PDF URL, or upload your own with `files.upload()`.

In [ ]:
!wget -q https://arxiv.org/pdf/1706.03762.pdf -O attention.pdf
!ls -lh attention.pdf

**Want to use your own PDF instead?** Uncomment and run this cell:

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# pdf_path = list(uploaded.keys())[0]

pdf_path = "attention.pdf"

---
## Step 3: Load the PDF — `Document Loaders`

**Concept:** LangChain has loaders for every format (PDF, HTML, Markdown, Word, websites, databases, etc.). Each loader returns a list of `Document` objects, where each `Document` has:
- `page_content`: the text
- `metadata`: where it came from (page number, source, etc.) — important for citations later

We use `PyPDFLoader` which splits a PDF page-by-page.

In [ ]:
!pip install --upgrade --force-reinstall numpy scipy


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"📄 Loaded {len(documents)} pages")
print(f"\n--- Sample (page 1, first 500 chars) ---")
print(documents[0].page_content[:500])
print(f"\n--- Metadata ---")
print(documents[0].metadata)


---
## Step 4: Split into Chunks — `Text Splitters`

**Why split?** Two reasons:
1. **LLM context limits** — even with huge context windows, sending a 50-page PDF for every question is wasteful and expensive
2. **Retrieval precision** — if your chunk is a whole page, the relevant 2 sentences get diluted by surrounding text

**Key parameters:**
- `chunk_size`: characters per chunk. Smaller = more precise retrieval, but might cut context mid-thought. Larger = more context per chunk but less precise. **1000 is a common sweet spot.**
- `chunk_overlap`: characters shared between adjacent chunks. Prevents losing context at chunk boundaries. **200 (20% of chunk_size) is typical.**

**Why `RecursiveCharacterTextSplitter`?** It tries to split on natural boundaries first (paragraphs → sentences → words → characters), keeping semantic chunks intact. This usually outperforms naive fixed-size splitting.

**Try changing chunk_size to 500 or 2000 later and see how retrieval quality changes — this is a real engineering decision in production RAG.**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # try paragraph breaks first
)

chunks = splitter.split_documents(documents)

print(f"✂️  Split into {len(chunks)} chunks")
print(f"\n--- Sample chunk #5 ---")
print(chunks[5].page_content)
print(f"\nMetadata: {chunks[5].metadata}")

---
## Step 5: Create Embeddings — Turning Text into Vectors

**Concept:** An embedding model converts text into a vector (a list of ~384–1024 numbers). The crucial property: **texts with similar meaning produce similar vectors**.

Example:
- "What is attention?" → `[0.21, -0.43, 0.88, ...]`
- "How does the attention mechanism work?" → `[0.19, -0.41, 0.90, ...]` ← very close
- "What's the weather today?" → `[-0.55, 0.71, 0.02, ...]` ← far away

**Why local embeddings?** Embedding millions of chunks via API gets expensive. Local models like `BAAI/bge-small-en-v1.5` are:
- Free
- Fast on GPU (Colab T4 is plenty)
- High quality — bge-small consistently beats OpenAI's `text-embedding-ada-002` on benchmarks despite being 100x smaller

**The downloaded model is ~130MB** — first run takes a minute, then it's cached.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}  # important for cosine similarity
)

# Test it: embed two similar and one different sentence
test_vectors = embeddings.embed_documents([
    "The cat sat on the mat",
    "A feline rested on the rug",
    "Python is a programming language"
])

import numpy as np
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\n'cat on mat' vs 'feline on rug':  {cosine_sim(test_vectors[0], test_vectors[1]):.3f}")
print(f"'cat on mat' vs 'python language': {cosine_sim(test_vectors[0], test_vectors[2]):.3f}")
print("\n☝️  Similar sentences score higher — this is what powers retrieval")

---
## Step 6: Store in a Vector Database — `Chroma`

**Concept:** Embedding 100 chunks once is fine. Embedding them on every query is slow and expensive. A vector database:
1. Stores chunks + their embeddings
2. Provides fast similarity search (find the top-k most similar vectors to a query)
3. Persists to disk so you don't re-embed every restart

**Why Chroma?** It's open-source, runs in-process (no separate server), and matches what the IBM lab uses. In production, you'd consider Pinecone, Weaviate, Qdrant, or pgvector — same concepts.

**What `from_documents` does:**
1. Calls the embedding model on every chunk
2. Stores `(chunk_text, embedding_vector, metadata)` tuples
3. Builds an index for fast similarity search

In [ ]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="attention_paper"
)

print(f"✅ Stored {vectordb._collection.count()} chunks in Chroma")

---
## Step 7: Test Retrieval — Does it Find Relevant Chunks?

Before adding an LLM, let's check that retrieval works. If retrieval is bad, no LLM can save you — the LLM only sees what retrieval gives it.

**This step is critical for debugging RAG.** If your final answers are wrong, the first question is *always*: "did retrieval surface the right chunks?"

In [ ]:
query = "What is multi-head attention?"

retrieved = vectordb.similarity_search(query, k=3)

print(f"🔍 Query: {query}\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Result {i} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:300])
    print()

---
## Step 8: Set Up the LLM — Gemini

**Concept:** This is the generation half of "Retrieval-Augmented Generation." The LLM takes the retrieved chunks + the user's question and writes a coherent answer.

**Why `temperature=0`?** For RAG, we want factual answers grounded in the documents, not creative writing. Low temperature → more deterministic, less hallucination.

**Model choice:** `gemini-1.5-flash` is free-tier friendly and fast. For higher quality, swap to `gemini-1.5-pro` (also free, but tighter rate limits).

In [ ]:
!pip install -q --upgrade google-generativeai

import os
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai

# Ensure the API key is configured for the google.generativeai client
# It should already be in os.environ from cell AjmC4ffeIPlB, but configure it for genai client directly
if "GOOGLE_API_KEY" in os.environ:
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
else:
    print("Warning: GOOGLE_API_KEY not found in environment. Please ensure it's set.")

# List available models and filter for those supporting 'generateContent'
print("Searching for available Gemini models...")
available_models_for_chat = [
    m.name for m in genai.list_models() if "generateContent" in m.supported_generation_methods
]

print(f"Found {len(available_models_for_chat)} models supporting 'generateContent':")
for model_name in available_models_for_chat:
    print(f"- {model_name}")

chosen_model_name = None
# Prioritize specific Gemini models for chat
preferred_models = ["gemini-1.0-pro", "gemini-pro", "gemini-1.5-flash", "gemini-1.5-pro"]

for p_model in preferred_models:
    # Check if the full model name (e.g., 'models/gemini-1.0-pro') is in the list
    if f"models/{p_model}" in available_models_for_chat:
        chosen_model_name = p_model
        print(f"Automatically selected preferred model: {chosen_model_name}")
        break

if not chosen_model_name and available_models_for_chat:
    # If preferred models not found, pick the first available one
    # Remove the 'models/' prefix for langchain_google_genai
    chosen_model_name = available_models_for_chat[0].replace("models/", "")
    print(f"Warning: No preferred Gemini models found. Using first available chat model: {chosen_model_name}")
elif not chosen_model_name:
    print("Error: No models supporting 'generateContent' found. Please check your API key, region, or available models.")
    llm = None # Indicate failure
    raise ValueError("No suitable Gemini model found for chat.")


llm = ChatGoogleGenerativeAI(
    model=chosen_model_name,
    temperature=0,
    max_output_tokens=1024,
)

# Quick test
try:
    test_response = llm.invoke("Say 'RAG is ready' in exactly three words").content
    print(f"LLM test successful with '{chosen_model_name}': {test_response}")
except Exception as e:
    print(f"Error during quick test with model {chosen_model_name}: {e}")
    print("Please ensure the selected model is suitable for your region/project and API key.")
    llm = None # Indicate failure

---
## Step 9: Build the RAG Chain

**Concept:** A chain ties retrieval + LLM together. We use LangChain Expression Language (LCEL) — the modern, composable way — instead of the older `RetrievalQA` class.

**What the chain does on each query:**
1. Send query to retriever → get top 4 chunks
2. Format chunks + query into a prompt
3. Send prompt to LLM → get answer
4. Return answer + the source chunks (for citation)

**The prompt template** is where you control behavior — instructing the model to only use the context, admit when it doesn't know, etc. This is *the* place where you reduce hallucinations.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# The retriever: pulls top-4 most similar chunks for any query
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

# The prompt: this is where you control hallucinations
prompt = ChatPromptTemplate.from_template("""You are a helpful assistant answering questions about a document.

Use ONLY the following context to answer the question. If the context doesn't contain the answer, say "I don't know based on the provided document." — do not make up information.

Context:
{context}

Question: {question}

Answer:""")

# Helper to format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(f"[Page {d.metadata.get('page', '?')}] {d.page_content}" for d in docs)

# The chain: retrieve → format → prompt → LLM → text output
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built")

---
## Step 10: Ask Questions! 🎉

Let's test it. Try the suggested queries, then ask your own.

In [ ]:
answer = rag_chain.invoke("What is the Transformer architecture and why did the authors propose it?")
print(answer)

In [ ]:
answer = rag_chain.invoke("How does multi-head attention work? Explain step by step.")
print(answer)

In [ ]:
# Test the hallucination guard — ask something NOT in the paper
answer = rag_chain.invoke("What is the price of Bitcoin today?")
print(answer)

---
## Step 11: Show Sources (Citations)

**Why this matters:** In production RAG, users (and regulators) want to know *where* an answer came from. Always return source chunks alongside the answer.

Here we extend the chain to return both the answer and the source documents.

In [ ]:
from langchain_core.runnables import RunnableParallel

rag_chain_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=(
        {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
        | prompt
        | llm
        | StrOutputParser()
    )
)

result = rag_chain_with_sources.invoke("What datasets did the authors evaluate on?")

print("📝 ANSWER:")
print(result["answer"])
print("\n📚 SOURCES:")
for i, doc in enumerate(result["context"], 1):
    print(f"  [{i}] Page {doc.metadata.get('page', '?')}: {doc.page_content[:120]}...")